In [1]:
from

In [1]:
import sys
import os
import torch
import numpy as np
from pathlib import Path

# Set up paths
os.chdir(os.path.abspath('..'))
%load_ext autoreload
%autoreload 2

# Calculate the project root directory
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
print(f"Adding {project_root} to sys.path")

# Add to sys.path -> necessary for importing the src package
if project_root not in sys.path:
    sys.path.append(project_root)

# Import necessary modules
import hydra
from omegaconf import OmegaConf
from hydra.core.global_hydra import GlobalHydra
from hydra import initialize, compose
from geofm_src.factory import create_dataset, create_model, model_registry, dataset_registry

# Initialize Hydra
# Clear any previous initialization
GlobalHydra.instance().clear()
# Initialize with the correct config path
initialize(config_path="../geofm_src/configs/", caller_stack_depth=2)

# Print available models in the registry
print("Available models in the registry:")
for model_name in model_registry.keys():
    print(f"- {model_name}")

# Print available datasets in the registry
print("\nAvailable datasets in the registry:")
for dataset_name in dataset_registry.keys():
    print(f"- {dataset_name}")

Adding /home/ando to sys.path


/home/ando/.conda/envs/fm/lib/python3.10/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


Available models in the registry:
- croma
- dinov2
- softcon
- dofa
- senpamae
- panopticon
- galileo
- anysat

Available datasets in the registry:
- geobench
- resisc45
- benv2
- corine
- digital_typhoon
- tropical_cyclone
- hyperview
- fmow
- dummy
- spacenet1


/data/tmp/ipykernel_2005685/2916192492.py:31: UserWarning: 
The version_base parameter is not specified.
Please specify a compatability version level, or None.
Will assume defaults for version 1.1
  initialize(config_path="../geofm_src/configs/", caller_stack_depth=2)


In [14]:
from geofm_src.datasets.spacenet1_wrapper import SpaceNet1BenchmarkDataset

In [22]:
ds_train = SpaceNet1BenchmarkDataset(root="/data/panopticon/datasets/spacenet1", split="train", image='8band')
ds_val = SpaceNet1BenchmarkDataset(root="/data/panopticon/datasets/spacenet1", split="val", image='8band')
ds_test = SpaceNet1BenchmarkDataset(root="/data/panopticon/datasets/spacenet1", split="test", image='8band')

/data/panopticon/datasets/spacenet1/SN1_buildings/train
/data/panopticon/datasets/spacenet1/SN1_buildings/val
/data/panopticon/datasets/spacenet1/SN1_buildings/test


In [20]:
len(ds_train), len(ds_val), len(ds_test)

(5552, 694, 694)

In [24]:
sample, mask = ds_val[400]
sample.shape, mask.shape

(torch.Size([8, 102, 110]), torch.Size([102, 110]))

In [26]:
sample

tensor([[[  62.,   54.,   46.,  ...,   89.,   22.,   42.],
         [  35.,   68.,   88.,  ...,   91.,   35.,   84.],
         [  33.,   49.,   87.,  ...,   96.,   54.,   22.],
         ...,
         [  97.,  165.,   68.,  ...,   86.,  100.,  100.],
         [ 154.,  100.,  135.,  ...,   78.,   75.,   91.],
         [  77.,   84.,   77.,  ...,   95.,   99.,  177.]],

        [[  79.,   87.,   87.,  ...,   98.,   93.,   21.],
         [  50.,   54.,   60.,  ...,   83.,   87.,   79.],
         [  21.,   33.,   67.,  ...,   60.,   60.,   44.],
         ...,
         [ 158.,  143.,  143.,  ...,  110.,  100.,   50.],
         [ 100.,   98.,   93.,  ...,   98.,  100.,  110.],
         [  87.,   79.,   90.,  ...,   67.,  149.,  221.]],

        [[  82.,  100.,  152.,  ...,  141.,  174.,   17.],
         [  56.,   93.,  107.,  ...,   79.,   95.,   87.],
         [  31.,   58.,   80.,  ...,   62.,   62.,   18.],
         ...,
         [ 465.,  451.,  431.,  ...,  303.,  247.,   42.],
         [

In [1]:
#!/usr/bin/env python3
"""
Download SpaceNet Dataset from AWS S3

This script demonstrates how to download the SpaceNet dataset from AWS S3 using Python's boto3 library
instead of the AWS CLI, with the ability to specify a custom download location.
"""

import boto3
import os
import sys
import threading
from botocore import UNSIGNED
from botocore.client import Config

# Progress tracking class
class ProgressPercentage(object):
    def __init__(self, client, bucket, key):
        self._size = client.head_object(Bucket=bucket, Key=key)['ContentLength']
        self._seen_so_far = 0
        self._lock = threading.Lock()
        
    def __call__(self, bytes_amount):
        with self._lock:
            self._seen_so_far += bytes_amount
            percentage = (self._seen_so_far / self._size) * 100
            sys.stdout.write(f"\rDownload progress: {percentage:.2f}% ({self._seen_so_far}/{self._size} bytes)")
            sys.stdout.flush()

def download_spacenet(download_dir='.', show_progress=True):
    """
    Download SpaceNet dataset to a specified directory
    
    Args:
        download_dir (str): Directory where the file will be downloaded
        show_progress (bool): Whether to show download progress
    
    Returns:
        str: Path to the downloaded file
    """
    # Create an S3 client with unsigned configuration (for public datasets)
    s3_client = boto3.client('s3', config=Config(signature_version=UNSIGNED))
    
    # Define the bucket and object key
    bucket_name = 'spacenet-dataset'
    object_key = 'spacenet/SN1_buildings/tarballs/SN1_buildings_train_AOI_1_Rio_3band.tar.gz'
    object_key = 'spacenet/SN1_buildings/tarballs/SN1_buildings_train_AOI_1_Rio_8band.tar.gz'
    # object_key = 'spacenet/SN1_buildings/tarballs/SN1_buildings_train_AOI_1_Rio_geojson_buildings.tar.gz'
    object_key = 'spacenet/SN1_buildings/tarballs/SN1_buildings_test_AOI_1_Rio_3band.tar.gz'


    filename = 'SN1_buildings_train_AOI_1_Rio_3band.tar.gz'
    filename = 'SN1_buildings_train_AOI_1_Rio_8band.tar.gz'
    # filename = 'SN1_buildings_train_AOI_1_Rio_geojson_buildings.tar.gz'
    filename = 'SN1_buildings_test_AOI_1_Rio_3band.tar.gz'

    # aws s3 cp s3://spacenet-dataset/spacenet/SN1_buildings/tarballs/SN1_buildings_train_AOI_1_Rio_8band.tar.gz .
    
    # Create the directory if it doesn't exist
    os.makedirs(download_dir, exist_ok=True)
    
    # Define the full local file path where the dataset will be saved
    local_file_path = os.path.join(download_dir, filename)
    
    print(f"Downloading {object_key} from {bucket_name}...")
    print(f"File will be saved to: {os.path.abspath(local_file_path)}")
    
    # Download the file with or without progress tracking
    if show_progress:
        s3_client.download_file(
            bucket_name, 
            object_key, 
            local_file_path,
            Callback=ProgressPercentage(s3_client, bucket_name, object_key)
        )
        print("\n")  # Add a newline after progress bar
    else:
        s3_client.download_file(bucket_name, object_key, local_file_path)
    
    print(f"Download complete! File saved to {os.path.abspath(local_file_path)}")
    return local_file_path

download_spacenet('/data/panopticon/datasets/spacenet1')
 

File will be saved to: /data/panopticon/datasets/spacenet1/SN1_buildings_test_AOI_1_Rio_3band.tar.gz
Download progress: 32.33% (349175808/1079959064 bytes)

KeyboardInterrupt: 

# SpaceNet Wrapper

In [5]:
import sys
import os

os.chdir(os.path.abspath('..'))
%load_ext autoreload
%autoreload 2

# Calculate the project root directory
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
print(f"Adding {project_root} to sys.path")

# Add to sys.path -> necessary for importing the src package
if project_root not in sys.path:
    sys.path.append(project_root)


import hydra
from omegaconf import OmegaConf
from hydra.core.global_hydra import GlobalHydra
from hydra.experimental import initialize, compose
from geofm_src.factory import create_dataset


# Set the config directory for OmegaConf to resolve relative paths
config_dir = 'geofm_src/configs/dataset'
# Note: if you run this multiple times, you need to clear the previous initialization
GlobalHydra.instance().clear()

# Initialize with the correct config path
initialize(config_path="../geofm_src/configs/dataset")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Adding /home to sys.path


/home/ando/.conda/envs/fm/lib/python3.10/site-packages/hydra/experimental/initialize.py:43: UserWarning: hydra.experimental.initialize() is no longer experimental. Use hydra.initialize()
  deprecation_warning(message=message)
/home/ando/.conda/envs/fm/lib/python3.10/site-packages/hydra/experimental/initialize.py:45: UserWarning: 
The version_base parameter is not specified.
Please specify a compatability version level, or None.
Will assume defaults for version 1.1
  self.delegate = real_initialize(


hydra.experimental.initialize()

In [10]:
ds_cfg = compose(config_name="spacenet1_8b.yaml")

train_ds, val_ds, test_ds = create_dataset(ds_cfg)
display(ds_cfg)

/data/panopticon/datasets/spacenet1/SN1_buildings/train
/data/panopticon/datasets/spacenet1/SN1_buildings/val
/data/panopticon/datasets/spacenet1/SN1_buildings/test


/home/ando/.conda/envs/fm/lib/python3.10/site-packages/hydra/experimental/compose.py:25: UserWarning: hydra.experimental.compose() is no longer experimental. Use hydra.compose()
  deprecation_warning(message=message)
/home/ando/.conda/envs/fm/lib/python3.10/site-packages/hydra/_internal/defaults_list.py:251: UserWarning: In 'spacenet1_8b.yaml': Defaults list is missing `_self_`. See https://hydra.cc/docs/1.2/upgrades/1.0_to_1.1/default_composition_order for more information
  warnings.warn(msg, UserWarning)


{'dataset_type': 'spacenet1', 'task': 'classification', 'num_classes': 2, 'num_channels': 8, 'data_path': '${oc.env:DATASETS_DIR}/spacenet1', 'wavelengths_mean_nm': [428, 482, 545, 604, 660, 723, 824, 906], 'wavelengths_sigma_nm': [18, 22, 27, 15, 23, 15, 42, 36], 'wavelengths_mean_microns': None, 'multilabel': False, 'image_resolution': 224, 'subset': {'train': -1, 'val': -1, 'test': -1}, 'input_key': None, 'sequence_length': None, 'band_gsds': [1, 1, 1, 1, 1, 1, 1, 1], 'scale': 1.0, 'normalize_target': True, 'senpamae_channels': [0, 1, 2, 3, 4, 5, 6, 7], 'senpamae_srf_name': 'rfs_wv23_recon.npy', 'dataset_name': 'fmow_wv23'}

In [13]:
import random
sample, mask = train_ds[random.randint(0, len(train_ds))]

sample.shape, mask.shape

(torch.Size([8, 224, 224]), torch.Size([224, 224]))

In [17]:
import random
sample, mask = val_ds[random.randint(0, len(val_ds))]

sample.shape, mask.shape

(torch.Size([8, 224, 224]), torch.Size([224, 224]))

In [20]:
import random
sample, mask = test_ds[random.randint(0, len(val_ds))]

sample.shape, mask.shape

(torch.Size([8, 224, 224]), torch.Size([224, 224]))

In [29]:
from torchgeo.datasets import Airphen

In [32]:
ds_train = Airphen(root="/data/panopticon/datasets/airphen", split="train", image='8band', download=True)
# ds_val = SpaceNet1(root="/data/panopticon/datasets/spacenet1", split="val", image='8band')

TypeError: RasterDataset.__init__() got an unexpected keyword argument 'root'